# 👻 Ghost Radar — Edisi Certainty Factor

Game menebak lokasi hantu dengan petunjuk sensor yang **kadang bohong**.
AI tidak tahu pasti di mana hantunya. Ia menyimpan **nilai keyakinan (Certainty Factor / CF)**
untuk setiap petak, lalu menggabungkan setiap petunjuk baru dengan **aturan kombinasi MYCIN**
ala sistem pakar. (Kategori: Uncertainty)

## Cara Membaca Papan

| Hal | Penjelasan |
|---|---|
| **Ukuran papan** | 4 × 4 = 16 petak (hanya satu level) |
| **Baris** | Dihitung dari atas ke bawah, mulai dari 0 (sumbu kiri) |
| **Kolom** | Dihitung dari kiri ke kanan, mulai dari 0 (sumbu bawah) |
| **Koordinat** | Selalu ditulis `baris kolom`. Contoh: `2 3` = baris 2, kolom 3 |
| **Angka di petak** | Nilai CF dari **-1** (hampir pasti bukan) sampai **+1** (hampir pasti ya). **0** = belum tahu |
| **Jumlah semua CF** | **Tidak** harus 100%. Tiap petak punya keyakinan sendiri |

## Arti Warna dan Garis

| Tampilan | Artinya |
|---|---|
| Merah pekat | Bukti kuat **mendukung** hantu ada di petak itu (CF mendekati +1) |
| Netral / gelap | Belum ada bukti (CF sekitar 0) |
| Biru / abu-abu | Bukti **menentang** (CF negatif) |
| Kotak garis **cyan** | Petak yang pernah di-scan, sensor berbunyi **"ada bisikan"** |
| Kotak garis **abu-abu** putus-putus | Petak yang pernah di-scan, sensor **"dingin"** (tidak berbunyi) |
| Tanda **X** | Petak yang sudah salah ditangkap (CF dipaksa jadi -1) |
| Kotak garis **hijau** (HANTU) | Posisi asli hantu, hanya ditampilkan saat game selesai |

## Aturan Permainan

- Hantu bersembunyi di satu petak dan **tidak pernah berpindah** selama game berlangsung.
- Awalnya CF semua petak = 0 (AI belum tahu apa-apa).
- **Scan** petak: butuh 1 baterai, lalu sensor memberi sinyal.
  - Hantu dekat (di petak itu atau 8 petak sekelilingnya): sensor berbunyi 80% kali.
  - Hantu jauh: sensor tetap bisa berbunyi 15% kali (**alarm palsu**).
  - Angka 80% dan 15% hanya mengatur perilaku sensor di dunia game. **AI tidak memakainya.**
- Setelah tiap scan, AI memberi **bukti CF** (ditetapkan oleh "pakar") ke setiap petak:

| Hasil sensor | Petak dekat petak yang di-scan | Petak jauh |
|---|---|---|
| Ada bisikan | +0.60 (mendukung) | -0.20 (sedikit menentang) |
| Dingin | -0.50 (menentang) | +0.10 (sedikit mendukung) |

- Bukti baru `b` digabung dengan CF lama `a` memakai rumus kombinasi MYCIN:
  - keduanya positif: `CF = a + b × (1 − a)`
  - keduanya negatif: `CF = a + b × (1 + a)`
  - tandanya berbeda: `CF = (a + b) / (1 − min(|a|, |b|))`
- Karena hantu diam, bukti lama **tidak dipudarkan**. Semua bukti terus menumpuk.
- **Tangkap** petak: kalau benar, menang. Kalau salah, nyawa berkurang 1 dan CF petak itu menjadi -1.
- Awal game: 3 nyawa dan 8 baterai. Nyawa habis = kalah.
- AI dianggap **cukup yakin** jika CF tertinggi ≥ 0.85 **dan** unggul minimal 0.15 dari petak lain.

## Cara Bermain

**Mode klik (Cell 8):** pilih aksi Scan/Tangkap lalu klik petak.

**Mode ketik (Cell 5):** jalankan `main_manual()` lalu ketik perintah di kotak input:

| Perintah | Fungsi | Contoh |
|---|---|---|
| `scan baris kolom` | Scan satu petak | `scan 1 2` |
| `tangkap baris kolom` | Coba tangkap hantu | `tangkap 0 3` |
| `keluar` | Berhenti dari game | `keluar` |

**Tips:**
1. Ikuti **saran scan** AI. AI memilih petak yang membelah kandidat teratas jadi dua bagian seimbang.
2. Jangan menangkap terlalu cepat. Tunggu sampai AI bilang **cukup yakin**.
3. Kalau banyak petak punya CF yang sama tinggi, scan petak yang memisahkan mereka.

Cell 1: Konfigurasi

In [ ]:
%matplotlib inline
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ====== PAPAN PERMAINAN (hanya 1 level) ======
UKURAN = 4            # papan 4 x 4 = 16 petak
BATERAI = 8           # jumlah scan maksimal
NYAWA = 3             # jumlah kesempatan salah tangkap

# ====== SIMULASI SENSOR (perilaku dunia game, TIDAK dipakai AI) ======
RADIUS = 1            # petak dianggap "dekat" jika jaraknya <= RADIUS
P_DEKAT = 0.80        # sensor berbunyi 80% kali jika hantu dekat petak yang di-scan
P_JAUH = 0.15         # sensor tetap berbunyi 15% kali jika hantu jauh -> alarm palsu

# ====== PARAMETER CERTAINTY FACTOR (skala 0 s/d 1) ======
CF_BISIK_DEKAT = 0.60    # sensor berbunyi -> petak dekat dapat bukti +0.60
CF_DINGIN_JAUH = 0.30    # sensor dingin   -> petak jauh dapat bukti +0.30
AMBANG_CF = 0.85         # CF tertinggi minimal segini agar AI "cukup yakin"
SELISIH_CF = 0.15        # ... dan harus unggul minimal segini dari petak lain

Cell 2: Inti AI (Certainty Factor + Saran Scan)

In [ ]:
def jarak(a, b):
    """Jarak Chebyshev: petak diagonal juga dihitung dekat."""
    return max(abs(a[0] - b[0]), abs(a[1] - b[1]))

def gabung_cf(a, b):
    """Rumus penggabungan dua bukti Certainty Factor: CF = CF1 + CF2 * (1 - CF1). Skala [0, 1]."""
    a = np.clip(np.asarray(a, dtype=float), 0.0, 1.0)
    b = np.clip(np.asarray(b, dtype=float), 0.0, 1.0)
    return np.clip(a + b * (1.0 - a), 0.0, 1.0)

def update_cf_dataset(unpicked, dataset, n=UKURAN):
    """Menghitung nilai CF berdasarkan grid yang belum dipilih saja dan kumpulan dataset."""
    cf = np.zeros((n, n))
    K = len(unpicked)
    if K == 0:
        return cf
    base = 1.0 / K
    for (r, c) in unpicked:
        cf[r, c] = base
    for (sr, sc, sinyal) in dataset:
        for (r, c) in unpicked:
            d = jarak((r, c), (sr, sc))
            e = (0.60 if d <= RADIUS else 0.08) if sinyal else (0.35 if d > RADIUS else 0.04)
            cf[r, c] = gabung_cf(cf[r, c], e)
    return cf

def tebakan_terbaik_cf(cf, terkunci=None):
    """Petak dengan CF tertinggi di antara petak yang belum terkunci."""
    n = cf.shape[0]
    terbaik_idx = (0, 0)
    terbaik_val = -1.0
    for r in range(n):
        for c in range(n):
            if terkunci is not None and terkunci[r, c]:
                continue
            if cf[r, c] > terbaik_val:
                terbaik_val = cf[r, c]
                terbaik_idx = (r, c)
    if terbaik_val < 0:
        terbaik_val = 0.0
    return terbaik_idx, float(terbaik_val)

def kandidat_cf(cf, terkunci=None):
    """Petak aktif yang CF-nya masih bersaing dengan CF tertinggi."""
    _, maks = tebakan_terbaik_cf(cf, terkunci)
    mask = (cf >= maks - SELISIH_CF)
    if terkunci is not None:
        mask = mask & (~terkunci)
    return mask

def ai_yakin(cf, terkunci=None):
    """AI cukup yakin jika CF tertinggi >= AMBANG_CF dan hanya tersisa satu kandidat."""
    _, nilai = tebakan_terbaik_cf(cf, terkunci)
    return nilai >= AMBANG_CF and int(kandidat_cf(cf, terkunci).sum()) == 1

def saran_scan_cf(cf, terkunci=None):
    """Heuristik scan AI: pilih petak belum terkunci yang membagi kandidat teratas seimbang."""
    n = cf.shape[0]
    kandidat = kandidat_cf(cf, terkunci)
    total = int(kandidat.sum())
    if total == 1:
        idx = np.unravel_index(np.argmax(kandidat), kandidat.shape)
        return (int(idx[0]), int(idx[1]))
    terbaik, skor_terbaik = (0, 0), -1
    for r in range(n):
        for c in range(n):
            if terkunci is not None and terkunci[r, c]:
                continue
            r0, r1 = max(0, r - RADIUS), min(n, r + RADIUS + 1)
            c0, c1 = max(0, c - RADIUS), min(n, c + RADIUS + 1)
            cov = int(kandidat[r0:r1, c0:c1].sum())
            skor = min(cov, total - cov)
            if skor > skor_terbaik:
                skor_terbaik, terbaik = skor, (r, c)
    return terbaik

Cell 3: Visualisasi Peta Certainty Factor

In [ ]:
from matplotlib.patches import Patch, Rectangle

def tampilkan_peta_cf(cf, riwayat=None, salah=None, hantu=None,
                      judul="Peta Certainty Factor (Radar Scanner)"):
    n = cf.shape[0]
    fig, ax = plt.subplots(figsize=(6.2, 5.8))
    fig.patch.set_facecolor("#070b14")
    ax.set_facecolor("#0b1320")
    im = ax.imshow(cf, cmap="magma", vmin=0, vmax=1)

    # nilai CF di tiap petak
    for r in range(n):
        for c in range(n):
            v = cf[r, c]
            ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=13,
                    color="white" if v >= 0.35 else "#94a3b8",
                    fontweight="bold" if v >= AMBANG_CF else "normal")

    # petak yang pernah di-scan
    if riwayat:
        for (r, c, sinyal) in riwayat:
            ax.add_patch(Rectangle((c - 0.46, r - 0.46), 0.92, 0.92, fill=False,
                                   edgecolor="#00f3ff" if sinyal else "#607d8b",
                                   lw=2.5, linestyle="-" if sinyal else ":"))

    # petak yang sudah salah ditangkap
    if salah:
        for (r, c) in salah:
            ax.text(c, r + 0.32, "X salah", ha="center", va="center",
                    fontsize=8, fontweight="bold", color="#ff2a5f")

    # posisi hantu (hanya saat game selesai)
    if hantu is not None:
        ax.add_patch(Rectangle((hantu[1] - 0.5, hantu[0] - 0.5), 1, 1,
                               fill=False, edgecolor="#00ffaa", lw=4))
        ax.text(hantu[1], hantu[0] - 0.32, "HANTU", ha="center", va="center",
                fontsize=8, fontweight="bold", color="#00ffaa")

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.tick_params(colors="#8faec9", labelsize=10)
    for spine in ax.spines.values():
        spine.set_color("#00f3ff")
        spine.set_linewidth(1.5)
    ax.set_xlabel("Kolom →", color="#00f3ff", fontweight="bold")
    ax.set_ylabel("Baris ↓", color="#00f3ff", fontweight="bold")
    ax.set_title(judul, color="#00ffaa", fontsize=13, fontweight="bold", pad=12)

    # legenda
    legenda = [
        Patch(facecolor="none", edgecolor="#00f3ff", linewidth=2.5, label="Scan: Ada Bisikan"),
        Patch(facecolor="none", edgecolor="#607d8b", linewidth=2, linestyle=":", label="Scan: Dingin"),
    ]
    if hantu is not None:
        legenda.append(Patch(facecolor="none", edgecolor="#00ffaa", linewidth=3.5,
                             label="Posisi Hantu"))
    leg = ax.legend(handles=legenda, loc="upper center", bbox_to_anchor=(0.5, -0.12),
                    ncol=3, fontsize=8, frameon=False)
    for text in leg.get_texts():
        text.set_color("#e2f1ff")

    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label("Certainty Factor (0 s/d 1)", color="#e2f1ff", fontsize=9)
    cbar.ax.yaxis.set_tick_params(color="#8faec9")
    plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="#8faec9")
    plt.show()

Cell 4: Kelas Game

In [ ]:
class GhostRadar:
    posisi_sebelumnya = None

    def __init__(self, seed=None):
        self.n = UKURAN
        self.rng = random.Random(seed)
        # posisi hantu acak sekali di awal, dan berbeda dari game sebelumnya
        while True:
            kandidat = (self.rng.randrange(self.n), self.rng.randrange(self.n))
            if GhostRadar.posisi_sebelumnya is None or kandidat != GhostRadar.posisi_sebelumnya:
                self.hantu = kandidat
                GhostRadar.posisi_sebelumnya = kandidat
                break
        self.cf = np.zeros((self.n, self.n))       # peta CF skala 0 s/d 1
        self.terkunci = np.zeros((self.n, self.n), dtype=bool)  # status kunci sel kosong
        self.ditemukan = False
        self.nyawa = NYAWA
        self.baterai = BATERAI
        self.riwayat = []          # (baris, kolom, sinyal)
        self.salah_tangkap = []
        self.dataset = []          # kumpulan data pemilihan grid
        self.status = "main"       # main / menang / kalah
        self.rehitung_cf()

    def get_unpicked(self):
        return [(r, c) for r in range(self.n) for c in range(self.n) if not self.terkunci[r, c]]

    def rehitung_cf(self):
        self.cf = update_cf_dataset(self.get_unpicked(), self.dataset, self.n)

    def scan(self, r, c):
        if self.status != "main":
            return "Game sudah selesai."
        if self.terkunci[r, c]:
            return "Petak ini sudah terkunci (kosong)."
        if self.baterai <= 0:
            return "Baterai habis! Kamu hanya bisa menangkap."
        self.baterai -= 1
        if (r, c) != self.hantu:
            self.cf[r, c] = 0.0
            self.terkunci[r, c] = True

        p = P_DEKAT if jarak(self.hantu, (r, c)) <= RADIUS else P_JAUH
        sinyal = self.rng.random() < p
        self.dataset.append((r, c, sinyal))
        self.rehitung_cf()
        self.riwayat.append((r, c, sinyal))
        return "Sinyal: ADA BISIKAN 👂" if sinyal else "Sinyal: dingin / tidak ada"

    def tangkap(self, r, c):
        if self.status != "main":
            return "Game sudah selesai."
        if self.terkunci[r, c]:
            return "Petak ini sudah terkunci (kosong)."
        if (r, c) == self.hantu:
            self.status = "menang"
            self.ditemukan = True
            self.cf[r, c] = 1.0
            return "🎉 BERHASIL! Hantu tertangkap."
        self.nyawa -= 1
        self.salah_tangkap.append((r, c))
        self.cf[r, c] = 0.0
        self.terkunci[r, c] = True
        self.dataset.append((r, c, False))
        self.rehitung_cf()
        if self.nyawa <= 0:
            self.status = "kalah"
            return "💀 Nyawa habis. Kamu kalah."
        return f"❌ Salah tangkap! Nyawa tersisa {self.nyawa}."

    # ----- bantuan AI (semuanya berbasis Certainty Factor) -----
    def tebakan(self):
        return tebakan_terbaik_cf(self.cf, self.terkunci)

    def saran(self):
        return saran_scan_cf(self.cf, self.terkunci)

    def yakin(self):
        return ai_yakin(self.cf, self.terkunci)

    def info(self):
        (tr, tc), nilai = self.tebakan()
        kandidat = int(kandidat_cf(self.cf, self.terkunci).sum())
        print(f"Nyawa: {self.nyawa} | Baterai: {self.baterai} | "
              f"Tebakan AI (CF): ({tr},{tc}) CF {nilai:.2f} | Kandidat: {kandidat}")
        if self.yakin():
            print("AI sudah cukup yakin, coba tangkap!")
        elif self.baterai > 0:
            print(f"Saran scan berikutnya: {self.saran()}")
        else:
            print("Baterai habis. Saatnya menangkap petak dengan CF tertinggi.")

Cell 5: Mode Manual (input dari pemain)

In [ ]:
from IPython.display import clear_output

def main_manual(seed=None):
    game = GhostRadar(seed=seed)
    pesan = "Game dimulai! Hantu bersembunyi (dan diam) di salah satu petak."
    while game.status == "main":
        clear_output(wait=True)
        print(pesan)
        tampilkan_peta_cf(game.cf, game.riwayat, game.salah_tangkap)
        game.info()
        cmd = input("Perintah (scan b k | tangkap b k | keluar): ").split()
        try:
            if cmd[0] == "keluar":
                break
            r, c = int(cmd[1]), int(cmd[2])
            if not (0 <= r < game.n and 0 <= c < game.n):
                pesan = "Koordinat di luar papan."
                continue
            if cmd[0] == "scan":
                pesan = game.scan(r, c)
            elif cmd[0] == "tangkap":
                pesan = game.tangkap(r, c)
            else:
                pesan = "Perintah tidak dikenal. Pakai: scan / tangkap / keluar"
        except (IndexError, ValueError):
            pesan = "Format salah. Contoh: scan 1 2"
    clear_output(wait=True)
    print(pesan, "| Status akhir:", game.status)
    tampilkan_peta_cf(game.cf, game.riwayat, game.salah_tangkap,
                      hantu=game.hantu, judul="Hasil Akhir")

# main_manual()   # <- jalankan ini untuk bermain

Cell 6: Mode Dataset (CSV)

In [ ]:
# Contoh dataset: riwayat scan (baris, kolom, sinyal 1/0) di papan 4x4
contoh = pd.DataFrame({
    "baris":  [1, 0, 0, 2],
    "kolom":  [1, 0, 2, 0],
    "sinyal": [1, 1, 0, 0],
})
contoh.to_csv("scan_contoh.csv", index=False)

def analisis_dataset(path_csv, n=UKURAN):
    """Membaca riwayat scan dari CSV, lalu menghitung CF berdasarkan grid belum dipilih dan dataset."""
    df = pd.read_csv(path_csv)
    riwayat, langkah = [], []
    terkunci = set()
    for i, row in df.iterrows():
        r, c, s = int(row["baris"]), int(row["kolom"]), bool(row["sinyal"])
        if not (0 <= r < n and 0 <= c < n):
            raise ValueError(f"Koordinat ({r},{c}) di luar papan {n}x{n}.")
        terkunci.add((r, c))
        riwayat.append((r, c, s))
        unpicked = [(x, y) for x in range(n) for y in range(n) if (x, y) not in terkunci]
        cf = update_cf_dataset(unpicked, riwayat, n)
        (tr, tc), nilai = tebakan_terbaik_cf(cf)
        langkah.append({
            "Langkah": i + 1,
            "Scan": f"({r},{c})",
            "Sinyal": "Ada bisikan" if s else "Dingin",
            "Sisa Belum Dipilih": len(unpicked),
            "Tebakan AI": f"({tr},{tc})",
            "CF tertinggi": round(nilai, 4),
            "Jumlah kandidat": int(kandidat_cf(cf).sum()),
            "AI yakin?": "Ya" if ai_yakin(cf) else "Belum",
        })
    display(pd.DataFrame(langkah))
    (tr, tc), nilai = tebakan_terbaik_cf(cf)
    print(f"Kesimpulan CF: hantu kemungkinan di ({tr},{tc}) dengan CF {nilai:.4f}")
    tampilkan_peta_cf(cf, riwayat, judul="Hasil Analisis Dataset (Certainty Factor)")
    return cf

_ = analisis_dataset("scan_contoh.csv")

Cell 7: Uji Cepat (Sanity Check)

In [ ]:
# 1. Uji rumus CF: CF = CF1 + CF2 * (1 - CF1) skala 0 s/d 1
assert abs(float(gabung_cf(0.6, 0.6)) - 0.84) < 1e-9
assert abs(float(gabung_cf(0.84, 0.30)) - 0.888) < 1e-9
print("Uji rumus CF lolos")

# 2. Uji update CF di papan 4x4
c2 = update_cf_dataset([(r, c) for r in range(UKURAN) for c in range(UKURAN) if (r, c) != (1, 1)], [(1, 1, True)])
assert c2.shape == (4, 4)
assert c2[1, 1] == 0.0                           # petak yang sudah dipilih terkunci 0
assert c2[0, 0] > 0.0 and c2[3, 3] > 0.0       # petak unpicked tidak ada yang bernilai 0
print("Uji update CF lolos")

# 3. Uji hantu tidak bergerak
g = GhostRadar(seed=7)
posisi_awal = g.hantu
while g.baterai > 0:
    g.scan(*g.saran())
assert g.hantu == posisi_awal
print(f"Uji hantu diam lolos: tetap di {posisi_awal} setelah {len(g.riwayat)} scan")

# 4. Uji contoh dataset (CSV scan_contoh.csv)
cf_res = analisis_dataset("scan_contoh.csv")
assert cf_res.shape == (4, 4)
    t_idx, t_val = tebakan_terbaik_cf(cf_res)
assert t_val > 0.0
print(f"Hasil tebakan dataset: {t_idx} dengan CF {t_val:.4f}")
print("Uji contoh dataset lolos")

# 5. Simulasi: AI Certainty Factor bermain sendiri
JUMLAH_GAME = 200

def simulasi_ai(seed=0):
    g = GhostRadar(seed=seed)
    while g.status == "main":
        if g.yakin() or g.baterai == 0:
            g.tangkap(*g.tebakan()[0])
        else:
            g.scan(*g.saran())
    return {"status": g.status, "scan": len(g.riwayat), "salah": len(g.salah_tangkap)}

hasil = [simulasi_ai(s) for s in range(JUMLAH_GAME)]
tabel = pd.DataFrame([{
    "Papan": f"{UKURAN}x{UKURAN}",
    "Metode": "Certainty Factor",
    "Jumlah game": JUMLAH_GAME,
    "Menang (%)": round(100 * sum(h["status"] == "menang" for h in hasil) / JUMLAH_GAME, 1),
    "Rata-rata scan": round(float(np.mean([h["scan"] for h in hasil])), 2),
    "Rata-rata salah tangkap": round(float(np.mean([h["salah"] for h in hasil])), 2),
}])
display(tabel)

Cell 8 : UI klik interaktif

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def _campur(c1, c2, t):
    """Campur dua warna RGB dengan porsi t (0..1)."""
    t = max(0.0, min(1.0, t))
    return "#" + "".join(f"{int(a + (b - a) * t):02x}" for a, b in zip(c1, c2))

def warna_cf(v):
    """CF skala 0 s/d 1: navy (0.0) -> ungu (0.5) -> merah neon (1.0)."""
    v = max(0.0, min(1.0, float(v)))
    if v >= 0.5:
        return _campur((139, 44, 196), (225, 29, 72), (v - 0.5) / 0.5)
    return _campur((11, 21, 36), (139, 44, 196), v / 0.5)


class GhostRadarUI:
    def __init__(self, seed=None):
        self.seed = seed
        self.game = None
        self.tombol = []

        self.btn_mulai = widgets.Button(
            description="Mulai Game Baru", button_style="success", icon="play",
            layout=widgets.Layout(width="180px"))
        self.mode = widgets.ToggleButtons(
            options=[("🔍 Scan (cari petunjuk)", "scan"),
                     ("🎯 Tangkap (tebak hantu)", "tangkap")],
            value="scan", description="Aksi:")
        self.btn_saran = widgets.Button(
            description="Saran Scan AI (CF)", button_style="warning", icon="lightbulb-o",
            layout=widgets.Layout(width="180px"))
        self.lbl_status = widgets.HTML()
        self.lbl_pesan = widgets.HTML()
        self.kotak_grid = widgets.VBox()

        self.penjelasan = widgets.HTML("""
        <div style='background:#0b1526; border:1px solid #1e3a5f; border-radius:8px; padding:10px 14px; margin:8px 0; color:#cbd5e1;'>
        <details style='font-size:13px;line-height:1.6'>
        <summary><b>ℹ️ Bagaimana AI Certainty Factor & Dataset berpikir? (klik untuk membuka)</b></summary>
        <p>Tiap petak punya <i>nilai keyakinan sendiri</i> dari 0.00 sampai 1.00 pada petak belum dipilih. Petak dicek terkunci 0.00; petak sekitar tetap bernilai positif.
        Kombinasi bukti memakai rumus: CF = CF1 + CF2 * (1 - CF1).
        jadi banyak petak boleh sama-sama tinggi.</p>
        <p>Setiap hasil scan diubah menjadi <b>bukti</b> oleh aturan pakar: ada bisikan memberi +0.60 ke
        petak dekat dan -0.20 ke petak jauh; dingin memberi -0.50 ke petak dekat dan +0.10 ke petak jauh.
        Bukti baru digabung dengan CF lama memakai <b>rumus kombinasi MYCIN</b>.</p>
        <p>Hantu tidak pernah berpindah, jadi semua bukti terus menumpuk. AI baru menyarankan
        menangkap jika CF tertinggi &ge; 0.85 <b>dan</b> unggul minimal 0.15 dari petak lain.</p>
        </details></div>""")

        self.legenda = widgets.HTML("""
        <div style='background:#070f1e; border:1px solid #1a2f4c; border-radius:8px; padding:10px 14px; font-size:12px; line-height:1.8; color:#94a3b8;'>
        <b>Cara membaca:</b> B = baris (atas ke bawah), K = kolom (kiri ke kanan).<br>
        <b>Angka di petak:</b> nilai CF 0.00 s/d 1.00; <span style='color:#e11d48;'>merah</span> = didukung bukti,
        <span style='color:#64748b;'>abu-biru</span> = ditentang bukti, gelap = belum tahu.<br>
        <span style='color:#00f0ff;'>Garis cyan</span> = scan "ada bisikan" &nbsp;|&nbsp;
        <span style='color:#94a3b8;'>garis abu-abu</span> = scan "dingin"
        &nbsp;|&nbsp; <span style='color:#f59e0b;'>garis oranye putus-putus</span> = saran AI<br>
        ✖ = sudah salah ditangkap &nbsp;|&nbsp; 👻 hijau = posisi hantu (muncul saat game selesai)
        </div>""")

        self.btn_mulai.on_click(self._mulai)
        self.btn_saran.on_click(self._saran)

    # ---------- tampilan awal ----------
    def tampil(self):
        panel = widgets.VBox([
            widgets.HTML("<h2 style='color:#00f0ff; margin-bottom:5px; font-family:sans-serif;'>👻 GHOST RADAR "
                         "<span style='font-size:14px; color:#94a3b8; font-weight:normal;'>— Certainty Factor 4x4</span></h2>"),
            widgets.HBox([self.btn_mulai, self.btn_saran]),
            self.mode,
            self.penjelasan,
            self.lbl_status,
            self.lbl_pesan,
            self.kotak_grid,
            self.legenda,
        ])
        display(panel)
        self._mulai()

    # ---------- membuat papan tombol ----------
    def _bangun_grid(self):
        n = self.game.n
        px = 88
        items = [widgets.HTML("<div style='text-align:center;font-weight:bold;color:#94a3b8;font-size:9px;'>B \\ K</div>", layout=widgets.Layout(width="68px", height="28px"))]
        for c in range(n):
            items.append(widgets.HTML(
                f"<div style='text-align:center;font-weight:800;color:#00f3ff;font-size:12px;line-height:28px;font-family:monospace;background:rgba(0,243,255,0.15);border:1px solid #00f3ff;border-radius:4px;'>KOLOM {c}</div>",
                layout=widgets.Layout(width=f"{px}px", height="28px")))
        self.tombol = []
        for r in range(n):
            items.append(widgets.HTML(
                f"<div style='text-align:center;font-weight:800;color:#00ffaa;font-size:11px;line-height:{px}px;font-family:monospace;background:rgba(0,255,170,0.15);border:1px solid #00ffaa;border-radius:4px;'>BARIS {r}</div>",
                layout=widgets.Layout(width="68px", height=f"{px}px")))
            baris = []
            for c in range(n):
                b = widgets.Button(
                    description="",
                    layout=widgets.Layout(width=f"{px}px", height=f"{px}px",
                                          padding="0px", margin="0px"))
                b.style.font_size = "16px"
                b.style.font_weight = "bold"
                b.on_click(lambda _b, r=r, c=c: self._klik(r, c))
                baris.append(b)
                items.append(b)
            self.tombol.append(baris)
        grid = widgets.GridBox(items, layout=widgets.Layout(
            grid_template_columns=f"68px repeat({n}, {px}px)", grid_gap="6px",
            padding="10px", border="1px solid #1a2f4c", width="fit-content"))
        self.kotak_grid.children = [grid]

    # ---------- memperbarui warna dan teks ----------
    def _render(self):
        g = self.game
        terakhir = {(r, c): s for (r, c, s) in g.riwayat}
        for r in range(g.n):
            for c in range(g.n):
                b = self.tombol[r][c]
                v = g.cf[r, c]
                if g.terkunci[r, c]:
                    b.style.button_color = "#0a0f18"
                    b.style.text_color = "#64748b"
                    b.description = "❌ 0.00"
                    b.layout.border = "1px solid #334155"
                    continue
                b.style.button_color = warna_cf(v)
                b.style.text_color = "#ffffff" if v > 0.3 else "#94a3b8"
                b.description = f"{v:.2f}"
                border = "1px solid #1a2f4c"
                if (r, c) in terakhir:
                    border = "3px solid #00f0ff" if terakhir[(r, c)] else "3px dotted #64748b"
                if (r, c) in g.salah_tangkap:
                    b.description = "✖"
                    border = "3px solid #ff2a5f"
                b.layout.border = border
        if g.status != "main":
            hr, hc = g.hantu
            self.tombol[hr][hc].description = "👻"
            self.tombol[hr][hc].layout.border = "4px solid #10b981"

        (tr, tc), nilai = g.tebakan()
        kandidat = int(kandidat_cf(g.cf, g.terkunci).sum())
        unpicked_count = len(g.get_unpicked())
        if tr <= 1 and tc <= 1: arah = "Barat Laut (B0-B1, K0-K1)"
        elif tr <= 1 and tc >= 2: arah = "Timur Laut (B0-B1, K2-K3)"
        elif tr >= 2 and tc <= 1: arah = "Barat Daya (B2-B3, K0-K1)"
        else: arah = "Tenggara (B2-B3, K2-K3)"
        if g.status == "main" and g.yakin():
            saran = "<span style='color:#ef4444;font-weight:bold;'>⚡ AI YAKIN! COBA TANGKAP!</span>"
        elif g.status == "main" and nilai >= AMBANG_CF:
            saran = (f"<span style='color:#f59e0b;font-weight:bold;'>CF sudah tinggi, tapi masih ada "
                     f"{kandidat} petak yang bersaing. Scan lagi untuk memisahkan.</span>")
        else:
            saran = f"<span style='color:#94a3b8;'>Belum yakin (butuh CF &ge; {AMBANG_CF:.2f} dan 1 kandidat).</span>"
        self.lbl_status.value = (
            "<div style='background:#070f1e; border:1px solid #1a2f4c; border-radius:8px; padding:10px 14px; "
            "font-size:14px; line-height:1.7; color:#e2e8f0; margin-bottom:6px;'>"
            f"❤️ Nyawa: <b>{g.nyawa}</b> &nbsp;&nbsp; 🔋 Baterai: <b>{g.baterai}</b>/{BATERAI} &nbsp;&nbsp; Sisa Belum Dipilih: <b>{unpicked_count}</b><br>"
            f"🧭 Arah Kemungkinan Hantu: <b style='color:#f59e0b;'>{arah}</b><br>"
            f"🎯 Tebakan AI (CF): <b style='color:#00f0ff;'>(B{tr}, K{tc}) {nilai:.2f}</b> "
            f"&nbsp;|&nbsp; Kandidat bersaing: <b>{kandidat}</b><br>{saran}</div>")

    def _pesan(self, teks):
        self.lbl_pesan.value = f"<div style='color:#38bdf8; font-size:14px; padding:4px 0; font-family:monospace;'>🔔 <b>{teks}</b></div>"

    # ---------- aksi ----------
    def _mulai(self, _=None):
        self.game = GhostRadar(seed=self.seed)
        self._bangun_grid()
        self._render()
        self._pesan("Game 4x4 dimulai! Hantu bersembunyi dan tidak akan berpindah. Pilih aksi, lalu klik petak.")

    def _klik(self, r, c):
        g = self.game
        if g is None:
            return
        if g.status != "main":
            self._pesan("Game sudah selesai. Tekan 'Mulai Game Baru'.")
            return
        if g.terkunci[r, c]:
            return
        hasil = g.scan(r, c) if self.mode.value == "scan" else g.tangkap(r, c)
        self._render()
        self._pesan(hasil)

    def _saran(self, _=None):
        g = self.game
        if g is None or g.status != "main":
            return
        if g.baterai <= 0:
            self._pesan("Baterai habis. Tangkap petak dengan CF tertinggi.")
            return
        r, c = g.saran()
        self._render()
        self.tombol[r][c].layout.border = "3px dashed #f59e0b"
        self._pesan(f"💡 Rekomendasi Scan AI (CF): Baris {r}, Kolom {c}.")


# ====== JALANKAN ======
ui = GhostRadarUI()      # untuk demo yang sama persis: GhostRadarUI(seed=42)
ui.tampil()

In [ ]:
# ==============================================================================
# GHOST RADAR - WEB EDITION (animasi radar + efek suara)
# ==============================================================================
# Catatan: beberapa editor (mis. VS Code) memblokir iframe ke file lokal.
# Jika kotak di bawah kosong, buka langsung lewat terminal: python app.py
from IPython.display import HTML, display

def tampilkan_radar_interaktif(tinggi=920):
    display(HTML(f'''
    <div style="background:#050914; padding:12px; border-radius:12px; border:1px solid #00f0ff; box-shadow:0 0 30px rgba(0,240,255,0.25); text-align:center; margin:10px 0;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px; padding:0 8px;">
            <span style="color:#00f0ff; font-weight:bold; font-family:monospace; letter-spacing:1.5px; font-size:14px;">⚡ GHOST RADAR - CERTAINTY FACTOR 4x4</span>
            <span style="background:rgba(0,240,255,0.15); color:#38bdf8; padding:3px 10px; border-radius:20px; font-size:11px; border:1px solid #00f0ff;">WEB AUDIO & RADAR SWEEP</span>
        </div>
        <iframe src="index.html" width="100%" height="{tinggi}px" style="border:1px solid #1a2f4c; border-radius:8px; background:#070b14;"></iframe>
    </div>
    '''))

tampilkan_radar_interaktif()